In [1]:
import pandas as pd
import numpy as np
# from scipy.signal import argrelextrema
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# from sklearn.linear_model import LinearRegression
# from datetime import datetime, date, timedelta, timezone, time
import talib.abstract as ta
# import seaborn as sns
# import matplotlib.pyplot as plt
import freqtrade.vendor.qtpylib.indicators as qtpylib
from freqtrade.strategy import merge_informative_pair
import os
from pathlib import Path
from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_data

In [2]:
project_root = "."
i=0
try:
    os.chdirdir(project_root)
    assert Path('.gitignore').is_file()
except:
    while i<4 and (not Path('.gitignore').is_file()):
        os.chdir(Path(Path.cwd(), '../'))
        i+=1
    project_root = Path.cwd()
print(Path.cwd())

/home/mo/Repositories/trade


In [ ]:
ticker = 'BTC'

config = Configuration.from_files(["user_data/rsi_cycle_engine.json"])
config["timeframe"] = "1h"
config["strategy"] = "RSICycleEngine"
data_location = config["datadir"]
pair = f"{ticker}/USDT:USDT"

backtest_dir = config["user_data_dir"] / "backtest_results"
trades = load_backtest_data(backtest_dir)
trades['color'] = np.where(trades.profit_abs >= 0, 'green', 'red')

base_url = "/home/mo/Repositories/trade/user_data/data/bybit/futures/"
# dataframe_15m = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-15m-futures.feather")
# dataframe_1h = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-1h-futures.feather")
# dataframe_4h = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-4h-futures.feather")
dataframe_1d = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-1d-futures.feather")

In [23]:
def extract_features(dataframe, c1, c2, col, name, direction="forward"):

    df = dataframe.copy()

    starts = df.loc[c1].reset_index().rename(columns={"index": "start"})[['start']]
    ends   = df.loc[c2].reset_index().rename(columns={"index": "end"})[['end']]

    if starts.empty or ends.empty:
        dataframe[name] = np.nan
        return dataframe

    pairs = pd.merge_asof(
        starts.sort_values("start"),
        ends.sort_values("end"),
        left_on="start",
        right_on="end",
        direction=direction,
    ).dropna()[["start", "end"]]

    if pairs.empty:
        dataframe[name] = np.nan
        return dataframe

    intervals = pd.IntervalIndex.from_arrays(pairs["start"], pairs["end"], closed="both")
    df["range_id"] = pd.cut(df.index, intervals)

    e = 'max' if col == 'high' else 'min'
    df[name] = df.groupby("range_id", observed=True)[col].transform(e)
    df[name] = np.where(df[col] == df[name], df[col], np.nan)

    return dataframe.merge(df[[name]], left_index=True, right_index=True, how="left")

In [24]:
def populate_features(dataframe):

    dataframe["rsi"] = ta.RSI(dataframe["close"], timeperiod=14)

    c1 = qtpylib.crossed_above(dataframe["rsi"], 70)
    c2 = qtpylib.crossed_below(dataframe["rsi"], 70)
    dataframe = extract_features(dataframe, c1, c2, "high", "max_high")
    dataframe = extract_features(dataframe, c2, c1, "low", "min_high")

    c1 = qtpylib.crossed_below(dataframe["rsi"], 30)
    c2 = qtpylib.crossed_above(dataframe["rsi"], 30)
    dataframe = extract_features(dataframe, c1, c2, "low", "min_low")
    dataframe = extract_features(dataframe, c2, c1, "high", "max_low")

    dataframe.loc[dataframe['max_high'].notna(),"cat"] = 'H'
    dataframe.loc[dataframe['min_low'].notna(),"cat"] = 'L'
    dataframe['cat'] = dataframe['cat'].ffill()

    return dataframe

In [35]:
def plot(dataframe, trades, row_heights=[0.55, 0.15, 0.15, 0.15], p1=[], p2=[], p3=[], p4=[]):
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.05
    )

    fig.add_trace(go.Candlestick(
        x=dataframe['date'],
        open=dataframe['open'],
        high=dataframe['high'],
        low=dataframe['low'],
        close=dataframe['close'],
        name='Price'
    ))
    
    for col, mode, color in p1:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=1, col=1
        )

    fig.add_trace(
        go.Scatter(
            x=trades.open_date,
            y=trades.open_rate,
            mode='markers',
            name="open date",
            marker=dict(
                color='orange',
                symbol="square-open",
                size=10,
                line=dict(width=3),
            ),
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=trades.close_date,
            y=trades.close_rate,
            mode='markers',
            name="trades",
            marker=dict(
                color=trades.color,
                symbol="square-open",
                size=10,
                line=dict(width=3),
            )
        ),
        row=1, col=1
    )

    for col, mode, color in p2:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=2, col=1
        )

    fig.add_hline(70, row=2, col=1)
    fig.add_hline(30, row=2, col=1)

    for col, mode, color in p3:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=3, col=1
        )

    fig.add_hline(70, row=3, col=1)
    fig.add_hline(30, row=3, col=1)

    for col, mode, color in p4:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=4, col=1
        )

    fig.add_hline(70, row=4, col=1)
    fig.add_hline(30, row=4, col=1)

    fig.update_layout(
        height=800, 
        showlegend=True,
        xaxis_rangeslider_visible=False
    )
    
    fig.show()

In [27]:
def populate_entry_trend(dataframe):

    dataframe.loc[
            (
                (qtpylib.crossed_above(dataframe["rsi"], 30))
            ),
            "enter_long"
        ] = dataframe['close']

    dataframe.loc[
            (
                (qtpylib.crossed_below(dataframe["rsi"], 70))
            ),
            "enter_short"
        ] = dataframe['close']

    return dataframe

In [29]:
# dataframe = dataframe_15m.copy()
# dataframe = populate_features(dataframe)

# dataframe = dataframe_1h.copy()
# dataframe = populate_features(dataframe)
# dataframe = merge_informative_pair(dataframe, informative, config["timeframe"], '1h', ffill=True)

dataframe = dataframe_1d.copy()
dataframe = populate_features(dataframe)
# dataframe = merge_informative_pair(dataframe, informative, config["timeframe"], '4h', ffill=True)

dataframe = populate_entry_trend(dataframe)

In [40]:
start = '2023-12-01'
end = '2025-04-30'
trades_red = trades.loc[
    (trades['pair'] == pair) & 
    (trades.open_date > start) & 
    (trades.open_date < end) &
    (trades.is_short == False)
]
data_red = dataframe.loc[(dataframe.date > start) & (dataframe.date < end)]

In [38]:
plot(
    dataframe,
    trades, 
    p1=[
        ('enter_short','markers','purple'),
    ], 
    p2=[
        (f'rsi','lines','blue'),
    ],
    # p3=[
    #     (f'rsi_1h','lines','blue'),
    # ],
    # p4=[
    #     (f'rsi_4h','lines','blue'),
    # ]
)

In [ ]:
# docker-compose run --rm atlas_engine_test backtesting --strategy AtlasEnginePlus --config user_data/atlas_engine_test.json --timerange 20231218- --export trades

In [13]:
import pandas as pd
import talib.abstract as ta

def compression_expansion(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    اضافه کردن ستون‌های فشردگی/انبساط بازار به دیتافریم
    """

    # محاسبه ATR (نوسان میانگین)
    dataframe['ATR'] = ta.ATR(dataframe, timeperiod=14)

    # محاسبه باند بولینگر
    bb = ta.BBANDS(dataframe, timeperiod=20, nbdevup=2.0, nbdevdn=2.0)
    dataframe['bb_upper'] = bb['upperband']
    dataframe['bb_middle'] = bb['middleband']
    dataframe['bb_lower'] = bb['lowerband']

    # عرض باند بولینگر (نسبی به میانگین)
    dataframe['bb_width'] = (dataframe['bb_upper'] - dataframe['bb_lower']) / dataframe['bb_middle']

    # معیار فشردگی: BB Width < 0.02 و ATR پایین‌تر از میانگین ATR
    dataframe['compression'] = (
        (dataframe['bb_width'] < 0.02) &
        (dataframe['ATR'] < dataframe['ATR'].rolling(20).mean())
    ).astype(int)

    # معیار انبساط: BB Width > 0.05 یا ATR بالاتر از میانگین ATR
    dataframe['expansion'] = (
        (dataframe['bb_width'] > 0.05) |
        (dataframe['ATR'] > dataframe['ATR'].rolling(20).mean())
    ).astype(int)

    return dataframe


In [16]:
dataframe = dataframe_4h.copy()
dataframe = compression_expansion(dataframe)

In [20]:
import pandas as pd
import numpy as np
import talib.abstract as ta
import plotly.graph_objects as go

df = dataframe_4h.copy()

# محاسبه پارابولیک SAR
df['SAR'] = ta.SAR(df, acceleration=0.02, maximum=0.2)

# رسم نمودار قیمت + نقاط SAR
fig = go.Figure()

# نمودار کندل‌استیک قیمت
fig.add_trace(go.Candlestick(
    x=df['date'],
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close'],
    name='Price'
))

# نقاط SAR
fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['SAR'],
    mode='markers',
    marker=dict(color='red', size=6),
    name='Parabolic SAR'
))

fig.update_layout(
    title='Parabolic SAR Indicator',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False
)

fig.show()
